In [ ]:
# Import Required Libraries
import os
from pathlib import Path
from dotenv import load_dotenv
from playwright.sync_api import sync_playwright
import re


In [ ]:
# Load Environment Credentials
repo_root = Path.cwd()
dotenv_path = repo_root / ".env"
load_dotenv(dotenv_path)

ADMIN_URL = os.getenv("ADMIN_URL", "http://127.0.0.1:8788/admin/profile")
ADMIN_USER = os.getenv("STUDENT_ADMIN_USER")
ADMIN_PASS = os.getenv("STUDENT_ADMIN_PASS")

print(f"Admin URL: {ADMIN_URL}")
print(f"Admin user: {ADMIN_USER}")


In [ ]:
# Configure Playwright Browser and Authentication

browser_args = [
    "--disable-dev-shm-usage",
    "--no-sandbox",
]

context_options = {
    "viewport": {"width": 1280, "height": 900},
    "ignore_https_errors": True,
}

if not ADMIN_USER or not ADMIN_PASS:
    raise RuntimeError("Admin credentials missing in .env")


In [ ]:
# Navigate to Ket Chips Site and Authenticate

def run_check():
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True, args=browser_args)
        context = browser.new_context(**context_options)
        page = context.new_page()

        login_url = ADMIN_URL
        page.goto(login_url, wait_until="networkidle")

        # Attempt to login if the login form exists
        if page.query_selector("#loginForm"):
            page.fill("#loginUser", ADMIN_USER)
            page.fill("#loginPass", ADMIN_PASS)
            page.click("#loginBtn")
            page.wait_for_load_state("networkidle")

        return browser, context, page

browser, context, page = run_check()
print("Navigated to admin page")


In [ ]:
# Scan Site Elements for #fbffff Text Color

chip_selector = ".level-chip.panelbg-key, .level-chip.panelbg-key.level-chip-standard"
matcher = re.compile(r"^rgba\(255,\s*255,\s*255,\s*1\)$|^#fbffff$|^rgb\(255,\s*255,\s*255\)$", re.IGNORECASE)

all_chips = page.query_selector_all(chip_selector)
print(f"Found {len(all_chips)} KET chips")

mismatches = []
for index, chip in enumerate(all_chips, start=1):
    color = page.evaluate("(element) => getComputedStyle(element).color", chip)
    text = page.evaluate("(element) => element.textContent.trim()", chip)
    if not matcher.search(color):
        mismatches.append({"index": index, "text": text, "color": color})

print("MISMATCHES:", mismatches)
print("Passed" if not mismatches else "Failed")


In [ ]:
# Report Results and Close Browser

if mismatches:
    print(f"{len(mismatches)} KET chip(s) have unexpected text color:")
    for mismatch in mismatches:
        print(mismatch)
else:
    print("All KET chips matched #fbffff text color.")

context.close()
browser.close()
